In [1]:
import os
import random
import torch
import numpy as np
import pandas as pd
import copy

seed = 12345

os.environ['PYTHONHASHSEED']=str(seed)
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=8):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, kernel_size=1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, kernel_size=1, bias=False)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        assert kernel_size in (3, 7), 'Kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiBranchConv(nn.Module):
    def __init__(self, output_channels=16, attention=True):
        super(MultiBranchConv, self).__init__()
        
        self.branch1 = nn.Conv2d(in_channels=1, out_channels=output_channels, kernel_size=(1, 24))
        self.branch2 = nn.Conv2d(in_channels=1, out_channels=output_channels, kernel_size=(2, 24))
        self.branch3 = nn.Conv2d(in_channels=1, out_channels=output_channels, kernel_size=(3, 24))
        self.branch4 = nn.Conv2d(in_channels=1, out_channels=output_channels, kernel_size=(4, 24))
        
        self.attn = attention
        self.ca1 = ChannelAttention(output_channels)
        self.ca2 = ChannelAttention(output_channels)
        self.ca3 = ChannelAttention(output_channels)
        self.ca4 = ChannelAttention(output_channels)
        self.sa1 = SpatialAttention(kernel_size=7)
        self.sa2 = SpatialAttention(kernel_size=7)
        self.sa3 = SpatialAttention(kernel_size=7)
        self.sa4 = SpatialAttention(kernel_size=7)

    def forward(self, x):

        # Branch 1: No padding needed
        out1 = F.relu(self.branch1(x))

        # Branch 2: Pad to the right (end)
        x_pad2 = F.pad(x, (0, 0, 0, 1))  # Padding only at the end (on the height dimension)
        out2 = F.relu(self.branch2(x_pad2))

        # Branch 3: Pad one row at the beginning and one at the end
        x_pad3 = F.pad(x, (0, 0, 1, 1))  # One padding at the start and one at the end
        out3 = F.relu(self.branch3(x_pad3))

        # Branch 4: Pad two rows at the beginning and one at the end
        x_pad4 = F.pad(x, (0, 0, 1, 2))  # One at the start, Two at the end
        out4 = F.relu(self.branch4(x_pad4))

        # Apply attention if enabled
        if self.attn:
            out1 = out1 * self.ca1(out1)
            out1 = out1 * self.sa1(out1)

            out2 = out2 * self.ca2(out2)
            out2 = out2 * self.sa2(out2)

            out3 = out3 * self.ca3(out3)
            out3 = out3 * self.sa3(out3)

            out4 = out4 * self.ca4(out4)
            out4 = out4 * self.sa4(out4)

        # Remove last dimension of size 1 (from Conv2D)
        out1 = out1.squeeze(-1)
        out2 = out2.squeeze(-1)
        out3 = out3.squeeze(-1)
        out4 = out4.squeeze(-1)

        # Transpose for concatenation later
        out1 = out1.transpose(1, 2)
        out2 = out2.transpose(1, 2)
        out3 = out3.transpose(1, 2)
        out4 = out4.transpose(1, 2)

        # Concatenate along the last dimension
        output = torch.cat((out1, out2, out3, out4), dim=-1)
        
        return output


In [5]:
import torch
import torch.nn as nn
import math

class CRISPRTransformerModel(nn.Module):
    def __init__(self, config):
        super(CRISPRTransformerModel, self).__init__()
        
        # Model parameters
        self.input_dim = 64
        self.num_layers = config.get("num_layers", 2)
        self.num_heads = config.get("num_heads", 8)
        self.dropout_prob = config["dropout_prob"]
        self.number_hidden_layers = config["number_hidder_layers"]
        self.seq_length = config.get("seq_length", 24)
        
        
        # Positional encoding
        self.pos_encoder = nn.Parameter(torch.randn(1, self.seq_length, self.input_dim))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.input_dim,
            nhead=self.num_heads,
            dim_feedforward=self.input_dim * 4,
            dropout=self.dropout_prob,
            batch_first=True,
            norm_first=True  
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=self.num_layers
        )
        
        # Convolutional preprocessing
        self.conv = MultiBranchConv(attention=config["attn"])
        
        # Hidden layers
        self.hidden_layers = []
        start_size = self.seq_length*self.input_dim
        for i in range(self.number_hidden_layers):
            layer = nn.Sequential(
                nn.Linear(start_size, start_size // 2),
                nn.GELU(),  
                nn.Dropout(self.dropout_prob)
            )
            self.hidden_layers.append(layer)
            start_size = start_size // 2
        self.hidden_layers = nn.ModuleList(self.hidden_layers)
        
        # Output layer
        self.output = nn.Linear(start_size, 2)

    def forward(self, x, src_mask=None):
        # Apply conv layer
        x = self.conv(x)  # Shape: [batch_size, seq_len, input_dim]
        
        # Add positional encoding
        x = x + self.pos_encoder
        
        # Apply transformer encoder
        x = self.transformer_encoder(x)
        
        x = x.view(x.size(0), -1)
        
        # Apply hidden layers
        for layer in self.hidden_layers:
            x = layer(x)
        
        x = self.output(x)
        
        return x

In [6]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class TrainerDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = torch.tensor(inputs, dtype=torch.float32).unsqueeze(1)  # Add channel dimension
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [7]:
import torch
from torch.utils.data import DataLoader

def train_model(model, train_loader, optimizer, scheduler, criterion, device, num_epochs):
    model = model.to(device)
    history = {'train_loss': []}
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        # Training loop
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Loss tracking
            train_loss += loss.item()
        
        # Average loss for the epoch
        train_loss /= len(train_loader)
        history['train_loss'].append(train_loss)
        
        print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f}")
        scheduler.step()
    
    return model, history

In [8]:
def trainer(config, train_x, train_y):
    seed = 12345
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    train_dataset = TrainerDataset(train_x, train_y)
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)

    model = CRISPRTransformerModel(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    class_weights = torch.tensor([1.0, config['pos_weight']]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    trained_model, history = train_model(model, train_loader, optimizer, scheduler, criterion, device, config['epochs'])
    return trained_model, history

In [9]:
def tester(model, test_x, test_y):
    test_dataset = TrainerDataset(test_x, test_y)
    test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    model.eval()
    results = []
    true_labels = []
    with torch.no_grad():
        for test_features, test_labels in test_dataloader:
            outputs = model(test_features.to(device)).detach().to("cpu")
            results.extend(outputs)
            true_labels.extend(test_labels)
    return true_labels, results

In [10]:
class Stats:
    def __init__(self):
        self.acc = 0
        self.pre = 0
        self.re = 0
        self.f1 = 0
        self.roc = 0
        self.prc = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        self.tp = 0
    def print(self):
        print('Accuracy: %.4f' %self.acc)
        print('Precision: %.4f' %self.pre)
        print('Recall: %.4f' %self.re)
        print('F1 Score: %.4f' %self.f1)
        print('ROC: %.4f' %self.roc)
        print('PR AUC: %.4f' %self.prc)
        print("Confusion Matrix")
        print(self.tn, "\t", self.fp)
        print(self.fn, "\t", self.tp)

In [11]:
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import auc
from sklearn.metrics import accuracy_score

def eval_matrices(model, test_x, test_y, debug = True):
    true_y, results = tester(model, test_x, test_y)
    predictions = [torch.nn.functional.softmax(r) for r in results]
    pred_y = np.array([y[1].item() for y in predictions])
    pred_y_list = []
    test_y = np.array([y.item() for y in true_y])

    for x in pred_y:
        if(x>0.5):
            pred_y_list.append(1)
        else:
            pred_y_list.append(0)

    pred_y_list = np.array(pred_y_list)
    tn, fp, fn, tp = confusion_matrix(test_y, pred_y_list).ravel()
    precision, recall, _ = precision_recall_curve(test_y, pred_y)
    auc_score = auc(recall, precision)
    acc = accuracy_score(test_y, pred_y_list)

    pr = -1
    re = -1
    f1 = -1
    try:
        pr = tp / (tp+fp)
        re = tp / (tp+fn)
        f1 = 2*pr*re / (pr+re)
    except:
        f1 = -1

    stats = Stats()
    stats.acc = acc
    stats.pre = pr
    stats.re = re
    stats.f1 = f1
    stats.roc = roc_auc_score(test_y, pred_y)
    stats.prc = auc_score
    stats.tn = tn
    stats.fp = fp
    stats.fn = fn
    stats.tp = tp

    if debug:
        print('Accuracy: %.4f' %acc)
        print('Precision: %.4f' %pr)
        print('Recall: %.4f' %re)
        print('F1 Score: %.4f' %f1)
        print('ROC:',roc_auc_score(test_y, pred_y))
        print('PR AUC: %.4f' % auc_score)

        # print(classification_report(test_y, pred_y_list, digits=4))
        # print("Confusion Matrix")
        # print(confusion_matrix(test_y, pred_y_list))

    return stats

In [12]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

def load_and_split(file_path):
    print(f"Loading Dataset: {file_path}")
    
    # Resolving structural differences mathematically 
    df = pd.read_csv(file_path, header=None, names=['Target sgRNA', 'Off Target sgRNA', 'label'])
    
    # Ensuring Float->Int precision mappings mapping
    df['label'] = df['label'].astype(int)
    
    Positive = df[df['label'] == 1]
    Negative = df[df['label'] == 0]
    
    Train_Negative, Test_Negative = train_test_split(Negative, test_size=0.15, random_state=42)
    Train_Positive, Test_Positive = train_test_split(Positive, test_size=0.15, random_state=42)
    
    train_data = pd.concat([Train_Negative, Train_Positive]).sample(frac=1, random_state=42).reset_index(drop=True)
    test_data = pd.concat([Test_Negative, Test_Positive]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    return train_data, test_data

def one_hot_features_bulge(df):
    orig_pairs = [f'{n1}{n2}' for n1 in ['A', 'T', 'G', 'C'] for n2 in ['A', 'T', 'G', 'C']]
    new_pairs = ['A-', 'T-', 'C-', 'G-', '-A', '-T', '-C', '-G']
    pairs = orig_pairs + new_pairs # Exaclty 24 elements for bulge mapping
    
    seq_length = len(df.iloc[0]['Target sgRNA'])
    pairwise_features = np.zeros((len(df), seq_length, 24))
    
    for idx, row in df.iterrows():
        on_seq = row['Target sgRNA'].replace('_', '-')
        off_seq = row['Off Target sgRNA'].replace('_', '-')
        
        for pos in range(seq_length):
            pair = on_seq[pos] + off_seq[pos]
            if pair in pairs:
                pairwise_features[idx, pos, pairs.index(pair)] = 1
                
    return pairwise_features, seq_length

In [13]:
configurations = [
    # Base Config (Best for I1 dataset)
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    
    # ---------------- I1 Optimizations (10 configs) ----------------
    # High capacity tweaks based on base config
    {'num_layers': 3, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    {'num_layers': 2, 'num_heads': 8, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    
    # Dropout / Regularization adjustments
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.15, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.25, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    
    # Pos-Weight / Loss adjustment
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 12, 'attn': True},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 8, 'attn': True},
    
    # Batch size & LR tuning
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 128, 'epochs': 40, 'learning_rate': 0.0002, 'pos_weight': 10, 'attn': True},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 32, 'epochs': 40, 'learning_rate': 0.00005, 'pos_weight': 10, 'attn': True},
    
    # Hidden Layer modification
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 2, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 0, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},

    # ---------------- I2 Optimizations (8 configs) ----------------
    # I2 often needs different regularization and attention sensitivity
    {'num_layers': 3, 'num_heads': 8, 'number_hidder_layers': 2, 'dropout_prob': 0.25, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.3, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 15, 'attn': True}, # Higher pos_weight for I2 class unbalance
    {'num_layers': 4, 'num_heads': 8, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 128, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 12, 'attn': True},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': False}, # Try without attn
    {'num_layers': 2, 'num_heads': 8, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 32, 'epochs': 40, 'learning_rate': 0.00005, 'pos_weight': 8, 'attn': True},
    {'num_layers': 3, 'num_heads': 4, 'number_hidder_layers': 2, 'dropout_prob': 0.15, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': False},
    {'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 2, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0002, 'pos_weight': 5, 'attn': True},
    {'num_layers': 2, 'num_heads': 2, 'number_hidder_layers': 1, 'dropout_prob': 0.1, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True},
]

In [14]:
datasets = ['I1.txt', 'I2.txt']

In [15]:
from sklearn.model_selection import train_test_split

for dset in datasets:
    file_path = f'/kaggle/input/datasets/anamulhoqueemtiaj/crispr/{dset}'
    if not os.path.exists(file_path):
        continue

    train_data, test_data = load_and_split(file_path) 
    train_x, seq_len = one_hot_features_bulge(train_data)
    test_x, _ = one_hot_features_bulge(test_data)
    
    train_y = train_data['label'].to_numpy()
    test_y = test_data['label'].to_numpy()
    print(f'------Traing with Dataset {dset}-------')
    for config in configurations:
        seed = 12345
        os.environ['PYTHONHASHSEED']=str(seed)
        random.seed(seed)
        torch.manual_seed(seed)
        np.random.seed(seed)
        torch.cuda.manual_seed_all(seed)
        print(f'Traing with dataset {dset} for config:')
        print(config)
        model, history = trainer(config, train_x, train_y)
        stats = eval_matrices(model, test_x, test_y)

Loading Dataset: /kaggle/input/datasets/anamulhoqueemtiaj/crispr/I1.txt


------Traing with Dataset I1.txt-------
Traing with dataset I1.txt for config:
{'num_layers': 2, 'num_heads': 4, 'number_hidder_layers': 1, 'dropout_prob': 0.2, 'batch_size': 64, 'epochs': 40, 'learning_rate': 0.0001, 'pos_weight': 10, 'attn': True}


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1/40 | Train Loss: 0.2605


Epoch 2/40 | Train Loss: 0.1378


...

Epoch 3/40 | Train Loss: 0.1253


Epoch 4/40 | Train Loss: 0.1166


Epoch 5/40 | Train Loss: 0.1099


Epoch 6/40 | Train Loss: 0.1047


Epoch 7/40 | Train Loss: 0.0993
